In [0]:
# Databricks Notebook: silver_common_utils
from pyspark.sql.functions import *
from delta.tables import DeltaTable

CATALOG_NAME = "data_dev_olist"
SCHEMA_NAME = "silver"

# Tự động tạo Schema nếu chưa có
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")

def run_silver_pipeline(
    source_folder_name: str,   
    table_name: str,           
    merge_key: str,            
    transform_func=None,       
    sync_delete=True           
):
    print(f"🚀 Khởi chạy pipeline cho: {source_folder_name} -> {table_name}")
    
    # 1. PATH CONFIGURATION
    source_path = f"abfss://raw-data@quocluudata.dfs.core.windows.net/bronze_delta/{source_folder_name}"
    silver_path = f"abfss://raw-data@quocluudata.dfs.core.windows.net/silver/{table_name}"
    target_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table_name}"

    # 2. READ BRONZE
    df_bronze = spark.read.format("delta").load(source_path)

    # 3. TRANSFORM & DEDUPLICATE
    processed_df = df_bronze.na.drop()
    if transform_func:
        processed_df = transform_func(processed_df)
        
    processed_df = processed_df.dropDuplicates([merge_key])
    processed_df = (
        processed_df
        .withColumn("is_active", lit(True))
        .withColumn("_processed_at", current_timestamp())
    )

    # 4. KIỂM TRA THƯ MỤC VẬT LÝ TRÊN ADLS
    is_path_exists = False
    try:
        dbutils.fs.ls(silver_path)
        is_path_exists = True
    except:
        is_path_exists = False

    # 5. EXECUTE WRITE OR MERGE
    if not is_path_exists:
        print(f"  ✨ Thư mục vật lý chưa tồn tại hoặc rỗng. Tiến hành khởi tạo bảng: {target_table}...")
        (
            processed_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .option("path", silver_path)
            .saveAsTable(target_table)
        )
    else:
        print(f"  🔄 Thư mục vật lý đã tồn tại. Tiến hành MERGE (Upsert) dựa trên khóa [{merge_key}]...")
        silver_table = DeltaTable.forName(spark, target_table)
        (
            silver_table.alias("target")
            .merge(
                processed_df.alias("source"),
                f"target.{merge_key} = source.{merge_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    # 6. FIX LỖI SYNC SOFT DELETE: CHUYỂN SANG DÙNG SƠ ĐỒ KẾT HỢP ĐỂ TƯƠNG THÍCH MỌI PHIÊN BẢN
    if sync_delete:
        print("  🗑 Đang đối chiếu kiểm tra dữ liệu bị xóa từ file nguồn Bronze...")
        active_source_ids = processed_df.select(merge_key)
        silver_table = DeltaTable.forName(spark, target_table)
        
        try:
            # Cách 1: Thử dùng hàm gốc tương thích phiên bản cũ hơn
            (
                silver_table.alias("target")
                .merge(
                    active_source_ids.alias("source"),
                    f"target.{merge_key} = source.{merge_key}"
                )
                .whenNotMatchedBySourceUpdate(
                    update={"is_active": lit(False), "_processed_at": current_timestamp()}
                )
                .execute()
            )
        except AttributeError:
            # Cách 2: Nếu cả hàm trên cluster cũng quá cũ, dùng SQL thuần chạy trực tiếp trên Delta Table
            print("  ⚠️ Cluster đang chạy phiên bản cũ. Chuyển hướng xử lý bằng Spark SQL...")
            
            # Tạo một View tạm thời chứa các ID đang hoạt động ở Bronze
            active_source_ids.createOrReplaceTempView("current_bronze_ids_view")
            
            spark.sql(f"""
                MERGE INTO {target_table} AS target
                USING current_bronze_ids_view AS source
                ON target.{merge_key} = source.{merge_key}
                WHEN NOT MATCHED BY SOURCE AND target.is_active = true THEN
                  UPDATE SET is_active = false, _processed_at = CURRENT_TIMESTAMP()
            """)

    # 7. PRINT SUMMARY METRICS
    final_df = spark.table(target_table)
    print(f"  📊 Tổng số dòng hiện tại ở Silver: {final_df.count()}")
    print(f"  🟢 Số dòng đang Active         : {final_df.filter('is_active = true').count()}")
    if sync_delete:
        print(f"  🔴 Số dòng đã bị Soft Deleted  : {final_df.filter('is_active = false').count()}")
    print(f"✅ Hoàn thành bảng: {table_name}\n" + "-"*50)

In [0]:
# MAGIC %run ./silver_common_utils

from pyspark.sql.functions import col, round, md5, concat, lit

# ==============================================================================
# 1. ĐỊNH NGHĨA LOGIC BIẾN ĐỔI ĐẶC THÙ CHO TỪNG BẢNG
# ==============================================================================
def transform_products(df):
    cols_to_int = ["product_description_length", "product_photos_qty", "product_length_cm", "product_height_cm", "product_width_cm"]
    for c in cols_to_int:
        if c in df.columns:
            df = df.withColumn(c, col(c).cast("integer"))
    if "product_weight_g" in df.columns:
        df = df.withColumn("product_weight_g", col("product_weight_g").cast("integer"))
    return df

def transform_order_items(df):
    # Tạo khóa tổ hợp duy nhất: pk_hash = md5(order_id + order_item_id)
    df = df.withColumn("pk_hash", md5(concat(col("order_id"), col("order_item_id"))))
    df = df.withColumn("price", round(col("price"), 2).cast("double")) \
             .withColumn("freight_value", round(col("freight_value"), 2).cast("double"))
    return df

def transform_payments(df):
    # Tạo khóa tổ hợp duy nhất: pk_hash = md5(order_id + payment_sequential)
    df = df.withColumn("pk_hash", md5(concat(col("order_id"), col("payment_sequential"))))
    df = df.withColumn("payment_value", round(col("payment_value"), 2).cast("double"))
    if "payment_installments" in df.columns:
        df = df.withColumn("payment_installments", col("payment_installments").cast("integer"))
    return df

def transform_reviews(df):
    if "review_comment_title" in df.columns:
        df = df.drop("review_comment_title")
    return df

def transform_geo(df):
    return df.filter(
        (col("geolocation_lat") <= 5.27438888)
        & (col("geolocation_lng") >= -73.98283055)
        & (col("geolocation_lat") >= -33.75116944)
        & (col("geolocation_lng") <= -34.79314722)
    )

# ==============================================================================
# 2. MẢNG CẤU HÌNH CHI TIẾT (Đã cập nhật đúng khóa chính duy nhất)
# ==============================================================================
tables_config = [
    {"source": "olist_customers", "target": "clean_customer", "key": "customer_id", "transform": None, "sync_delete": True},
    {"source": "olist_sellers", "target": "clean_seller", "key": "seller_id", "transform": None, "sync_delete": True},
    {"source": "olist_orders", "target": "clean_order", "key": "order_id", "transform": None, "sync_delete": True},
    {"source": "olist_products", "target": "clean_product", "key": "product_id", "transform": transform_products, "sync_delete": True},
    {"source": "olist_order_items", "target": "clean_order_item", "key": "pk_hash", "transform": transform_order_items, "sync_delete": True},
    {"source": "olist_order_payments", "target": "clean_payment", "key": "pk_hash", "transform": transform_payments, "sync_delete": True},
    {"source": "olist_order_reviews", "target": "clean_order_review", "key": "review_id", "transform": transform_reviews, "sync_delete": True},
    {"source": "olist_geolocation", "target": "clean_geolocation", "key": "geolocation_zip_code_prefix", "transform": transform_geo, "sync_delete": False},
    {"source": "product_category_name_translation", "target": "clean_product_category", "key": "product_category_name", "transform": None, "sync_delete": False}
]

# ==============================================================================
# 3. VÒNG LẶP TỰ ĐỘNG THỰC THI
# ==============================================================================
print(f"🔄 Đang quét danh sách thư mục trong 'bronze_delta' để xử lý...")
print("=" * 60)

for config in tables_config:
    try:
        run_silver_pipeline(
            source_folder_name=config["source"],
            table_name=config["target"],
            merge_key=config["key"],
            transform_func=config["transform"],
            sync_delete=config["sync_delete"]
        )
    except Exception as e:
        print(f"🔥 LỖI khi xử lý folder {config['source']}: {str(e)}")
        print("⏭️ Bỏ qua để chạy bảng tiếp theo...")
        print("=" * 60)

print("🏁 PIPELINE ĐÃ CHẠY HOÀN TẤT CHO CẢ TẬP DỮ LIỆU BRONZE DELTA!")

# ==============================================================================
# 4. XỬ LÝ ĐẶC THÙ CHO BẢNG DATE DIMENSION (DẪN XUẤT TỪ ORDERS)
# ==============================================================================
def transform_date_dimension(df):
    # Chỉ giữ lại cột timestamp, bỏ null và lấy giá trị duy nhất (distinct) giống code cũ
    return df.select("order_purchase_timestamp").na.drop().distinct()

try:
    print("📅 Bắt đầu khởi tạo/cập nhật bảng Date Dimension từ nguồn Orders...")
    
    run_silver_pipeline(
        source_folder_name="olist_orders",          # Đọc từ file nguồn Orders gốc
        table_name="date_dimension",                # Tên bảng đích ở tầng Silver
        merge_key="order_purchase_timestamp",       # Khóa chính để Merge
        transform_func=transform_date_dimension,    # Nạp logic lọc distinct timestamp
        sync_delete=False                           # Tắt Sync Delete vì đây là bảng Dim dẫn xuất
    )
    
except Exception as e:
    print(f"🔥 LỖI khi xử lý bảng Date Dimension: {str(e)}")

🔄 Đang quét danh sách thư mục trong 'bronze_delta' để xử lý...
🚀 Khởi chạy pipeline cho: olist_customers -> clean_customer
  ✨ Thư mục vật lý chưa tồn tại hoặc rỗng. Tiến hành khởi tạo bảng: data_dev_olist.silver.clean_customer...
  🗑 Đang đối chiếu kiểm tra dữ liệu bị xóa từ file nguồn Bronze...
🔥 LỖI khi xử lý folder olist_customers: DeltaMergeBuilder.whenNotMatchedBySourceUpdate() got an unexpected keyword argument 'update'
⏭️ Bỏ qua để chạy bảng tiếp theo...
🚀 Khởi chạy pipeline cho: olist_sellers -> clean_seller
  ✨ Thư mục vật lý chưa tồn tại hoặc rỗng. Tiến hành khởi tạo bảng: data_dev_olist.silver.clean_seller...
  🗑 Đang đối chiếu kiểm tra dữ liệu bị xóa từ file nguồn Bronze...
🔥 LỖI khi xử lý folder olist_sellers: DeltaMergeBuilder.whenNotMatchedBySourceUpdate() got an unexpected keyword argument 'update'
⏭️ Bỏ qua để chạy bảng tiếp theo...
🚀 Khởi chạy pipeline cho: olist_orders -> clean_order
  ✨ Thư mục vật lý chưa tồn tại hoặc rỗng. Tiến hành khởi tạo bảng: data_dev_olist.

In [0]:
%sql
 table data_dev_olist.silver.clean_customer;

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,is_active,_processed_at
237098a64674ae89babdc426746260fc,4390ddbb6276a66ff1736a6710205dca,82820,curitiba,PR,true,2026-06-13T16:17:27.225Z
e3109970a3fe8021d5ff82c577ce5606,a8654e2af5da6bb72f52c22b164855e1,5528,sao paulo,SP,true,2026-06-13T16:17:27.225Z
c532a74a3ebf1bacce2e2bcce3783317,91ec50a00ae74d0a229d2efdf4344e1e,14026,ribeirao preto,SP,true,2026-06-13T16:17:27.225Z
19cecb194f54e614b70d971306a9931b,d251c190ca75786e9ab937982d60d1d4,30320,belo horizonte,MG,true,2026-06-13T16:17:27.225Z
c82a5e4fafdbeb34f08928ccfba27d14,ca19a17e381182923b66007a351574b7,85854,foz do iguacu,PR,true,2026-06-13T16:17:27.225Z
b06429ef920fcfdd75713c712c9ee7b7,9316f45a5da8403a5938bd6069b1a4a7,12240,sao jose dos campos,SP,true,2026-06-13T16:17:27.225Z
d3ab15f0bd2c58865d566ab645572cd5,9ccfff93c79f3dd996cce15f26480c5b,21615,rio de janeiro,RJ,true,2026-06-13T16:17:27.225Z
031cd5f826be3d804771e3e3a1b21a1c,717aa48025662fcf27ddebbecc5f782b,41706,salvador,BA,true,2026-06-13T16:17:27.225Z
79adcf02229a33e78f0f5412a2434f53,8c8fccc50566baaed602e3775d9d5665,21515,rio de janeiro,RJ,true,2026-06-13T16:17:27.225Z
e3c7e245a96d7fa339fe6c16f8da4e90,79051ee5ee98c4bd6982e67e2e79dbcb,7847,franco da rocha,SP,true,2026-06-13T16:17:27.225Z


In [0]:
%sql 
select *from data_dev_olist.silver.clean_order_item

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,pk_hash,is_active,_processed_at
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29,99b9fecfdded739c9c7f03d24d133923,true,2026-06-13T16:17:40.753Z
000e906b789b55f64edcb1f84030f90d,1,57d79905de06d8897872c551bfd09358,ea8482cd71df3c1969d7b9473ff13abc,2017-11-27T19:09:02.000Z,21.99,11.85,90c017e75b4fe0a9d5366355562587b0,true,2026-06-13T16:17:40.753Z
001daeb0eddc45b999bad0801ad9d273,1,30c01cc81c9eb80469371743813789cc,f45122a9ab94eb4f3f8953578bc0c560,2017-05-25T02:15:20.000Z,38.33,16.79,ae7fe70e3a5a26349cb64c912db3c26d,true,2026-06-13T16:17:40.753Z
002b4e6fa42cd4a22cc86abc18fe9c05,1,a6f449f6257f26e556013151cee46b4b,b1a81260566c1bac3114a6d124413f27,2018-03-15T10:35:37.000Z,99.9,19.67,3b3f03f6fbe6e5498e84a5dc1db53289,true,2026-06-13T16:17:40.753Z
00335b686d693c7d72deeb12f8e89227,1,87b08e712cc4c9fe70984c5a24b29e2f,f00e21b1e91a79653163b7fd8f293ff1,2017-07-28T03:45:26.000Z,63.9,16.89,35a1a156a69f42258a87e2c18f6085b7,true,2026-06-13T16:17:40.753Z
00571ded73b3c061925584feab0db425,2,8695c431b31927efef5343e675f279e7,fe2032dab1a61af8794248c8196565c9,2017-05-25T21:10:16.000Z,179.9,15.01,2fd5e41fdd61e537504ec983904c73e0,true,2026-06-13T16:17:40.753Z
005a1dded353107dbabf8ffc83a20365,1,7e6c4a0bf900e259f50ba63331fd2785,6560211a19b47992c3666cc44a7e94c0,2018-02-15T18:10:50.000Z,75.0,14.28,7585fef7e90cdc9687f90c353a8431df,true,2026-06-13T16:17:40.753Z
005cad6157eadc7f1f09917607f1704a,1,7e97894cc00196a56d6ec315c68b2353,7008613ea464bad5cb9b83456e1e6a8f,2017-04-20T16:45:13.000Z,96.0,14.84,0029a89994521ce6dac20865b9ffaa6e,true,2026-06-13T16:17:40.753Z
006557c3221c1fcd02b0106343ab357b,1,7d187801885842645486f00fb166f378,d91fb3b7d041e83b64a00a3edfb37e4f,2018-03-01T14:51:13.000Z,14.9,8.27,2593ad9b3567ff7f47a94fa25140e242,true,2026-06-13T16:17:40.753Z
00685d31ae12e47470ba5c18ba74f22c,1,71bdf1a4d18c3cd68f108547f0b4cd1c,6fc26fe110feebd80a433e1f012a84f9,2017-08-31T21:55:16.000Z,59.0,16.17,3c9733420849859c67040eb37ce1fdfe,true,2026-06-13T16:17:40.753Z


In [0]:
%sql
DESCRIBE EXTENDED data_dev_olist.silver.clean_order_item;


col_name,data_type,comment
order_id,string,null
order_item_id,int,null
product_id,string,null
seller_id,string,null
shipping_limit_date,timestamp,null
price,double,null
freight_value,double,null
pk_hash,string,null
is_active,boolean,null
_processed_at,timestamp,null


In [0]:
display(
    dbutils.fs.ls(
        "abfss://raw-data@quocluudata.dfs.core.windows.net/silver/clean_customer"
    )
)

path,name,size,modificationTime
abfss://raw-data@quocluudata.dfs.core.windows.net/silver/clean_customer/_delta_log/,_delta_log/,0,1781367447000
abfss://raw-data@quocluudata.dfs.core.windows.net/silver/clean_customer/part-00000-3fd75adb-49b0-4180-a55d-a174288c328a.c000.snappy.parquet,part-00000-3fd75adb-49b0-4180-a55d-a174288c328a.c000.snappy.parquet,6917551,1781367448000


In [0]:
%sql 
DESCRIBE DETAIL data_dev_olist.silver.clean_customer;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,3cd496e6-1e11-4f01-bd4e-5b6bb4da8771,data_dev_olist.silver.clean_customer,null,abfss://raw-data@quocluudata.dfs.core.windows.net/silver/clean_customer,2026-06-13T16:17:26.989Z,2026-06-13T16:17:28.000Z,List(),List(),1,6917551,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql 
SHOW CREATE TABLE data_dev_olist.silver.clean_customer;

createtab_stmt
"CREATE TABLE data_dev_olist.silver.clean_customer ( customer_id STRING COLLATE UTF8_BINARY, customer_unique_id STRING COLLATE UTF8_BINARY, customer_zip_code_prefix INT, customer_city STRING COLLATE UTF8_BINARY, customer_state STRING COLLATE UTF8_BINARY, is_active BOOLEAN, _processed_at TIMESTAMP) USING delta LOCATION 'abfss://raw-data@quocluudata.dfs.core.windows.net/silver/clean_customer' TBLPROPERTIES ( 'delta.enableDeletionVectors' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7')"


In [0]:
%sql
SELECT * 
FROM delta.`abfss://raw-data@quocluudata.dfs.core.windows.net/silver/clean_customer`
LIMIT 10;


-- data_dev_olist.silver.clean_customer

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,is_active,_processed_at
237098a64674ae89babdc426746260fc,4390ddbb6276a66ff1736a6710205dca,82820,curitiba,PR,true,2026-06-13T16:17:27.225Z
e3109970a3fe8021d5ff82c577ce5606,a8654e2af5da6bb72f52c22b164855e1,5528,sao paulo,SP,true,2026-06-13T16:17:27.225Z
c532a74a3ebf1bacce2e2bcce3783317,91ec50a00ae74d0a229d2efdf4344e1e,14026,ribeirao preto,SP,true,2026-06-13T16:17:27.225Z
19cecb194f54e614b70d971306a9931b,d251c190ca75786e9ab937982d60d1d4,30320,belo horizonte,MG,true,2026-06-13T16:17:27.225Z
c82a5e4fafdbeb34f08928ccfba27d14,ca19a17e381182923b66007a351574b7,85854,foz do iguacu,PR,true,2026-06-13T16:17:27.225Z
b06429ef920fcfdd75713c712c9ee7b7,9316f45a5da8403a5938bd6069b1a4a7,12240,sao jose dos campos,SP,true,2026-06-13T16:17:27.225Z
d3ab15f0bd2c58865d566ab645572cd5,9ccfff93c79f3dd996cce15f26480c5b,21615,rio de janeiro,RJ,true,2026-06-13T16:17:27.225Z
031cd5f826be3d804771e3e3a1b21a1c,717aa48025662fcf27ddebbecc5f782b,41706,salvador,BA,true,2026-06-13T16:17:27.225Z
79adcf02229a33e78f0f5412a2434f53,8c8fccc50566baaed602e3775d9d5665,21515,rio de janeiro,RJ,true,2026-06-13T16:17:27.225Z
e3c7e245a96d7fa339fe6c16f8da4e90,79051ee5ee98c4bd6982e67e2e79dbcb,7847,franco da rocha,SP,true,2026-06-13T16:17:27.225Z


In [0]:
%sql 
-- Total Orders = DISTINCTCOUNT(fact_table[order_id])